notebook to download necessary ERA5, IFS HRES, graph cast, and pangu forecasts

In [1]:
#from scipy.io import netcdf
import numpy as np
import sys
import xarray as xr
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
import linearmodels as lm
import zarr

In [2]:
variables_s = ['2m_temperature','10m_wind_speed']

era5_path = (
    "gs://weatherbench2/datasets/era5/"
    "1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
)

hres_path = (
    "gs://weatherbench2/datasets/hres/"
    "2016-2022-0012-240x121_equiangular_with_poles_conservative.zarr"
)

graphcast_path = (
    "gs://weatherbench2/datasets/graphcast_hres_init/"
    "2020/date_range_2019-11-16_2021-02-01_12_hours-240x121_equiangular_with_poles_conservative.zarr"
)

pangu_path = (
    "gs://weatherbench2/datasets/pangu_hres_init/"
    "2020_0012_240x121_equiangular_with_poles_conservative.zarr"
)

hres_truth_path = (
    "gs://weatherbench2/datasets/hres/"  
    "2016-2022-0012-240x121_equiangular_with_poles_conservative.zarr"
)

midnights_ifs =list(range(4,41,4))
midnights =list(range(3,41,4))

start = '2020-01-01 00:00:00'
end = '2020-12-16 12:00:00'

In [ ]:

ERA5 = xr.open_zarr(
    era5_path,
    storage_options={"token": "anon"},
    consolidated=True,
)

# Select variables first
ERA5 = ERA5[variables_s[0]].sel(time=slice(start,'2020-12-26 12:00:00'))

ERA5 = ERA5.where(np.logical_or(ERA5['time.hour'] == 00, ERA5['time.hour'] == 12), drop=True)

temp_2m = ERA5

temp_2m.to_netcdf('era5_2m_temp_2020.nc')


#Cosine weights
cos_weights=np.cos(np.deg2rad(era5_temp.latitude))
cos_weights=np.tile(cos_weights/np.mean(cos_weights), len(era5_temp.time)*len(era5_temp.longitude))

In [ ]:

HRES = xr.open_zarr(
    hres_path,
    storage_options={"token": "anon"},
    consolidated=True,
)

HRES = HRES[variables_s[0]].sel( time=slice(start,end))

HRES = HRES.where(np.logical_or(HRES['time.hour'] == 00, HRES['time.hour'] == 12), drop=True)

HRES_temp_2m = HRES

In [ ]:

graphcast = xr.open_zarr(
    graphcast_path,
    storage_options={"token": "anon"},
    consolidated=True,
)

graphcast = graphcast[variables_s[0]].sel( time=slice(start,end))

graphcast = graphcast.where(np.logical_or(graphcast['time.hour'] == 00, graphcast['time.hour'] == 12), drop=True)

graphcast_temp_2m = graphcast

days = [24,72,120,168,240]


graphcast = graphcast.sel(
    prediction_timedelta=days
)

graphcast.to_netcdf('graphcast_2m_temp_2020.nc')

In [ ]:

pangu = xr.open_zarr(
    pangu_path,
    storage_options={"token": "anon"},
    consolidated=True,
)

pangu = pangu[variables_s[0]].sel( time=slice(start,end))

pangu = pangu.where(np.logical_or(pangu['time.hour'] == 00, pangu['time.hour'] == 12), drop=True)

pangu_temp_2m = pangu

days = [24,72,120,168,240]

pangu = pangu.sel(
    prediction_timedelta=days
)

pangu.to_netcdf('pangu_2m_temp_2020.nc')